# GStar Assignment 2 — Problem 1: GRPO on Countdown (Kaggle, 2× T4)

**First time: run a smoke test (about 20–30 min, interactive).** Set `SMOKE = True` and click *Run All*.
It runs every step at tiny size: the unit tests, zero-shot on 64 prompts, and 3 GRPO steps with 16 rollouts.
If it ends with `done:` and no traceback, set `SMOKE = False` for the real runs below.

**Before running**
1. *Settings → Accelerator → **GPU T4 ×2***, and *Settings → **Internet on*** (needs a phone-verified Kaggle account).
2. Set `RUN` in the next cell. Run this notebook **twice**, once per session (each session has its own 12 h):
   - **Session 1:** `RUN = "L256"` → zero-shot at L256 / L512 / L1024, then GRPO training at max 256 tokens.
   - **Session 2:** `RUN = "L512"` → GRPO training at max 512 tokens.
3. **W&B (optional):** *Add-ons → Secrets* → add a secret named `WANDB_API_KEY` containing your W&B key, and
   attach it to this notebook. Runs go to `WANDB_ENTITY/WANDB_PROJECT`, set in the next cell. Without the secret,
   the notebook still runs and logs only locally (TensorBoard + `metrics.jsonl`).
4. Use **Save Version → Save & Run All (Commit)**, so the run keeps going in the background for up to 12 h and the outputs are saved.
   Then download `outputs_<RUN>.zip` from the version's *Output* tab.

**What runs:** the notebook clones the GitHub repo and runs `kaggle/grpo_t4.py`, which imports the Problem-1
implementations from `starter.py` (Tasks 1–5) unchanged. A T4 can't hold the starter's single-A100 setup, so:
- vLLM rollouts run on GPU 1 in fp16;
- the policy (Qwen3-1.7B + LoRA adapters, frozen fp16 base) trains on GPU 0;
- the merged weights are pushed into vLLM before every rollout and every eval.

Other hyperparameters follow `starter.main()`: 80 steps, 128 rollouts per step, group size 8, clip 0.2, eval every 10 steps on the 1,024 test prompts.
Training stops early if the session's time budget runs out, then does a final eval and saves.

In [ ]:
import os, time, json, shutil, subprocess
SESSION_START = time.time()

SMOKE = False           # True = ~20-30 min pipeline test at tiny size (run this once first)
RUN = "L256"            # "L256" (session 1, also runs zero-shot) or "L512" (session 2)
LOSS_TYPE = "grpo"      # "grpo" or "dr_grpo"
RUN_ZEROSHOT = RUN == "L256"
REPO_URL = "https://github.com/huypg-zenai/GStar-Assignment-2-GRPO.git"
BRANCH = "main"
WANDB_ENTITY = "phunghuysoict-hust"
WANDB_PROJECT = "gstar-a2-grpo"
SESSION_LIMIT_MIN = 12 * 60   # Kaggle GPU session limit
SAFETY_MIN = 25               # kept free for the final eval, report and zip
MAX_TOKENS = int(RUN[1:])
OUT = "/kaggle/working/outputs"
REPO = "/kaggle/tmp/repo"     # outside /kaggle/working so the code isn't duplicated in the outputs
os.makedirs(OUT, exist_ok=True)
def minutes_left():
    return SESSION_LIMIT_MIN - (time.time() - SESSION_START) / 60
# W&B key from Kaggle Secrets (never hard-code it in the notebook)
WANDB_ARGS = ""
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    WANDB_ARGS = f"--wandb_project {WANDB_PROJECT} --wandb_entity {WANDB_ENTITY} --wandb_group {RUN}_{LOSS_TYPE}"
    print("W&B: on ->", WANDB_ENTITY + "/" + WANDB_PROJECT)
except Exception as e:
    print("W&B: off (no WANDB_API_KEY secret attached):", type(e).__name__)
print(RUN, LOSS_TYPE, "zero-shot:", RUN_ZEROSHOT)

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch
n = torch.cuda.device_count()
print("GPUs:", n, [torch.cuda.get_device_name(i) for i in range(n)])
if n < 2:
    print("WARNING: expected 2 GPUs (GPU T4 x2). With one GPU, vLLM and the policy share it and memory is tight.")

In [ ]:
if os.path.exists(REPO):
    shutil.rmtree(REPO)
!git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO}
commit = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"]).decode().strip()
print("commit", commit)
json.dump({"repo": REPO_URL, "commit": commit, "run": RUN, "loss_type": LOSS_TYPE}, open(f"{OUT}/run_info.json", "w"))

In [ ]:
# ~3-5 min. vLLM 0.8.5 pins torch 2.6.0; the scripts run in fresh subprocesses, so no kernel restart is needed.
!pip install -q -r {REPO}/kaggle/requirements_t4.txt 2>&1 | grep -vi "warning\|already satisfied" | tail -5
!python -c "import torch, vllm, transformers, peft; print('torch', torch.__version__, '| vllm', vllm.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__)"

In [ ]:
# Unit tests for Tasks 1-5 (PDF examples + edge cases), CPU only
!cd {REPO} && python -m pytest -q tests/test_tasks.py

In [ ]:
# Section 1.6.1: zero-shot baseline on the 1,024 test prompts (session 1 only, ~20-30 min)
if RUN_ZEROSHOT:
    !cd {REPO} && python kaggle/grpo_t4.py zeroshot --lengths 256 512 1024 {'--eval_n 64' if SMOKE else ''} --out {OUT}/zeroshot {WANDB_ARGS} --wandb_name zeroshot 2>&1 | grep -v "it/s" | tee {OUT}/zeroshot.log
    print(json.dumps(json.load(open(f"{OUT}/zeroshot/zeroshot.json")), indent=1))

In [ ]:
# Section 1.6.2: GRPO training (most of the session). Stops early if the time budget runs out.
BUDGET = int(minutes_left() - SAFETY_MIN)
RUN_DIR = f"{OUT}/train_L{MAX_TOKENS}_{LOSS_TYPE}"
print(f"time budget for training: {BUDGET} min -> {RUN_DIR}")
!cd {REPO} && python kaggle/grpo_t4.py train --max_tokens {MAX_TOKENS} --loss_type {LOSS_TYPE} --time_budget_min {BUDGET} {'--smoke' if SMOKE else ''} --out {RUN_DIR} {WANDB_ARGS} --wandb_name train_L{MAX_TOKENS}_{LOSS_TYPE} 2>&1 | grep -v "it/s" | tee {OUT}/train.log

In [ ]:
# Table + plots for this session (re-run report.py locally on both sessions' outputs for the combined table)
!cd {REPO} && python kaggle/report.py {OUT} --save {OUT}/report
from IPython.display import Image, Markdown, display
display(Markdown(open(f"{OUT}/report/results.md").read()))
for f in ["problem1_curves.png", "problem1_final_accuracy.png"]:
    if os.path.exists(f"{OUT}/report/{f}"):
        display(Image(f"{OUT}/report/{f}"))

In [ ]:
# Correct / partial / failed examples from the final eval (for the analysis section)
import glob
evals = sorted(glob.glob(f"{OUT}/train_*/eval_step*.jsonl"))
if evals:
    rows = [json.loads(l) for l in open(evals[-1])]
    for label, r in [("CORRECT", 1.0), ("PARTIAL", 0.1), ("FAILED", 0.0)]:
        ex = [x for x in rows if abs(x["reward"] - r) < 1e-8][:2]
        for x in ex:
            print(f"=== {label} | target {x['answer']['target']} numbers {x['answer']['numbers']} | eq {x['equation']} = {x['result']} | {x['output_tokens']} tokens")
            print(x["response"][-600:], "\n")

In [ ]:
shutil.make_archive(f"/kaggle/working/outputs_{RUN}", "zip", OUT)
print(f"saved /kaggle/working/outputs_{RUN}.zip; minutes used: {(time.time() - SESSION_START) / 60:.0f}")